# Scaffold Split Validation of Hybrid Random Forest QSAR Model

This notebook evaluates the ability of the Hybrid Random Forest QSAR model to generalize to previously unseen molecular scaffolds.

Unlike random train-test splitting, scaffold splitting separates molecules based on their core structural frameworks, providing a more realistic assessment of model performance in drug discovery applications.

The workflow includes:

- Bemis-Murcko scaffold generation
- Scaffold-based train-test splitting
- Hybrid feature generation
- Model training
- Performance evaluation
- Generalization assessment

In [1]:
import pandas as pd

df = pd.read_csv("final_12k_log_transformed_papp_dataset.csv")

df.head()

,canonical_smiles,standard_value,log_papp
0,Br.Cc1c2c(cc[n+]1Cc1ccccc1)c1ccc(OCC(=O)OCCCCO...,0.05,-1.301030
1,Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1,3.49,0.542825
2,Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1,5.90,0.770852
3,Brc1ccc(C2(CC3CCCC3)c3ccccc3-c3nccn32)cn1,25.00,1.397940
4,Brc1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1,31.00,1.491362


In [3]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import pandas as pd
import numpy as np

## Bemis-Murcko Scaffold Generation

Bemis-Murcko scaffolds are generated to represent the core structural framework of each molecule.

The scaffold retains the ring systems and connecting framework while removing peripheral substituents.

These scaffolds will be used to create scaffold-based train-test splits.

In [4]:
def get_scaffold(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol:
        return MurckoScaffold.MurckoScaffoldSmiles(
            mol=mol
        )

    return None

## Creating Scaffold Representations

The scaffold generation function is applied to all molecules in the dataset to identify the core structural framework associated with each compound.

In [5]:
df['scaffold'] = df['canonical_smiles'].apply(
    get_scaffold
)

df[['canonical_smiles', 'scaffold']].head()

,canonical_smiles,scaffold
0,Br.Cc1c2c(cc[n+]1Cc1ccccc1)c1ccc(OCC(=O)OCCCCO...,O=C(COc1ccc2c3cc[n+](Cc4ccccc4)cc3n(CCCc3ccccc...
1,Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1,c1ccc(-c2nc3ccccn3n2)cc1
2,Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1,c1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1
3,Brc1ccc(C2(CC3CCCC3)c3ccccc3-c3nccn32)cn1,c1cncc(C2(CC3CCCC3)c3ccccc3-c3nccn32)c1
4,Brc1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1,c1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1


## Exploring Scaffold Diversity

The number of unique molecular scaffolds is calculated to estimate the structural diversity present in the dataset.

Higher scaffold diversity generally indicates broader chemical space coverage.

print("Total Molecules:", len(df))

print("Unique Scaffolds:", df['scaffold'].nunique())

## Most Frequent Scaffolds

The most common scaffolds are identified to understand which structural frameworks are most strongly represented within the dataset.

In [7]:
df['scaffold'].value_counts().head(10)

scaffold
c1ccccc1                                                                         193
O=C(Nc1ccccn1)C1=CNC(Nc2nc3ccccc3o2)=NC1c1ccccc1                                  78
O=c1c2c([nH]c3ccccc13)CCCC2                                                       77
O=C(Nc1ccccc1)c1ccccc1                                                            59
O=c1[nH]cc(-c2ccccc2Oc2ccccc2)c2cc[nH]c12                                         49
O=C(Nc1ccccc1)Nc1ccnc2ccccc12                                                     43
O=C(NCCCN1CCCCC1)C1CCN(Cc2coc(-c3ccccc3)n2)CC1                                    34
O=c1[nH]c2n[nH]c(=S)n2c2ccccc12                                                   31
O=C(NCc1cn[nH]c1)C1=CNC(Nc2nc3ccccc3o2)=NC1c1ccccc1                               29
O=C1CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CN1     28
Name: count, dtype: int64

## Creating Scaffold-Based Train-Test Split

The dataset is split based on unique molecular scaffolds rather than individual molecules.

This ensures that molecules sharing the same core structural framework are assigned to only one subset.

As a result, the test set contains previously unseen scaffolds, providing a more realistic assessment of model generalization.

In [9]:
from sklearn.model_selection import train_test_split

## Splitting Unique Scaffolds

Unique scaffold identifiers are extracted and divided into training and testing scaffold groups using an 80:20 split ratio.

In [10]:
unique_scaffolds = df['scaffold'].unique()

train_scaffolds, test_scaffolds = train_test_split(
    unique_scaffolds,
    test_size=0.20,
    random_state=42
)

print("Training Scaffolds:", len(train_scaffolds))

print("Testing Scaffolds:", len(test_scaffolds))

Training Scaffolds: 5018
Testing Scaffolds: 1255


## Assigning Molecules to Training and Testing Sets

Molecules are assigned to either the training set or testing set based on their scaffold membership.

This guarantees that scaffold overlap between training and testing datasets is eliminated.

In [11]:
train_df = df[
    df['scaffold'].isin(train_scaffolds)
]

test_df = df[
    df['scaffold'].isin(test_scaffolds)
]

print("Training Molecules:", len(train_df))

print("Testing Molecules:", len(test_df))

Training Molecules: 9597
Testing Molecules: 2693


## Verifying Scaffold Separation

A validation step is performed to ensure that no molecular scaffold is shared between the training and testing datasets.

Successful separation confirms that the scaffold split is valid and free from scaffold leakage.

In [12]:
common_scaffolds = set(train_df['scaffold']).intersection(
    set(test_df['scaffold'])
)

print("Common Scaffolds:", len(common_scaffolds))

Common Scaffolds: 0


## Hybrid Molecular Feature Generation

Morgan fingerprints and physicochemical descriptors are generated separately for the scaffold-based training and testing datasets.

The resulting hybrid feature representation combines structural topology and physicochemical information for QSAR modeling.

In [13]:
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

def generate_hybrid_features(df):

    fingerprints = []
    descriptors_list = []

    for smi in df['canonical_smiles']:

        mol = Chem.MolFromSmiles(smi)

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=2048
        )

        fingerprints.append(np.array(fp))

        descriptors_list.append([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol)
        ])

    fp_array = np.array(fingerprints)

    desc_array = np.array(descriptors_list)

    X = np.hstack([fp_array, desc_array])

    return X

## Generating Hybrid Features for Training and Testing Sets

Hybrid molecular features are generated independently for the scaffold-based training and testing datasets.

In [16]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [17]:
X_train = generate_hybrid_features(train_df)

X_test = generate_hybrid_features(test_df)

y_train = train_df['log_papp'].values

y_test = test_df['log_papp'].values

print(X_train.shape)

print(X_test.shape)

(9597, 2054)
(2693, 2054)


## Training Hybrid Random Forest Model

The Hybrid Random Forest model is trained using scaffold-based training data.

Unlike previous experiments using random train-test splitting, the model is now evaluated on completely unseen molecular scaffolds.

In [18]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Generating Predictions

The trained Hybrid Random Forest model is applied to the scaffold-based test set to predict permeability values for previously unseen molecular scaffolds.

In [19]:
y_pred = rf_model.predict(X_test)

## Evaluating Scaffold Generalization Performance

Model performance is evaluated using:

- Root Mean Squared Error (RMSE)
- Coefficient of Determination (R²)

These metrics quantify the ability of the model to generalize to previously unseen scaffold classes.

In [20]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(
    y_test,
    y_pred
)

print("Scaffold Split RF RMSE:", rmse)

print("Scaffold Split RF R²:", r2)

Scaffold Split RF RMSE: 0.6565333270416209
Scaffold Split RF R²: 0.4980325719510992


# Final Conclusions: Scaffold Split Validation of Hybrid Random Forest QSAR Model

This notebook evaluated the generalization capability of the Hybrid Random Forest QSAR model using scaffold-based train-test splitting.

Unlike random train-test splitting, scaffold splitting separates molecules according to their Bemis-Murcko scaffolds, ensuring that molecules sharing the same core structural framework are not present in both training and testing datasets.

As a result, the test set contains previously unseen molecular scaffolds, providing a more realistic assessment of model performance in drug discovery applications.

---

## Scaffold Diversity Analysis

Dataset statistics:

- Total Molecules: 12,290
- Unique Scaffolds: 6,273

The high number of unique scaffolds indicates substantial structural diversity within the dataset and supports the use of scaffold-based validation.

Scaffold split statistics:

| Dataset | Number of Scaffolds | Number of Molecules |
|----------|----------|----------|
| Training Set | 5,018 | 9,597 |
| Testing Set | 1,255 | 2,693 |

Scaffold leakage verification:

| Validation Check | Result |
|----------|----------|
| Common Scaffolds Between Train and Test | 0 |

The absence of common scaffolds confirms a valid scaffold-based split and ensures an unbiased evaluation of model generalization.

---

## Hybrid Random Forest Performance

### Random Split Performance

| Metric | Value |
|----------|----------|
| RMSE | 0.530 |
| R² | 0.617 |

### Scaffold Split Performance

| Metric | Value |
|----------|----------|
| RMSE | 0.657 |
| R² | 0.498 |

### Performance Comparison

| Validation Method | RMSE | R² |
|----------|----------|----------|
| Random Split RF | 0.530 | 0.617 |
| Scaffold Split RF | 0.657 | 0.498 |

---

## Interpretation of Results

A decrease in predictive performance was observed when moving from random splitting to scaffold-based splitting.

This behavior is expected because scaffold splitting presents the model with completely unseen molecular frameworks, creating a substantially more challenging prediction task.

The performance reduction indicates that some predictive power originates from scaffold similarity present in random train-test splits.

However, the Hybrid Random Forest model maintained:

- R² ≈ 0.50
- acceptable predictive performance on unseen scaffolds

This suggests that the model learned meaningful structure-property relationships associated with permeability rather than simply memorizing scaffold patterns.

---

## Scientific Significance

Scaffold split validation provides a more realistic estimate of prospective model performance in drug discovery settings where new compounds often contain previously unseen chemotypes.

The results demonstrate that:

- random train-test splitting may overestimate model performance
- scaffold-based validation provides a stricter and more realistic assessment
- the Hybrid Random Forest model retains useful predictive capability on novel molecular scaffolds
- hybrid molecular representations capture transferable permeability-related information

---

## Overall Conclusion

The scaffold validation study demonstrated that the Hybrid Random Forest model possesses moderate but meaningful generalization capability beyond the scaffolds observed during training.

Although performance decreased relative to random splitting, the model maintained predictive power on previously unseen molecular frameworks, indicating that it learned chemically relevant permeability patterns.

Combined with:

- model benchmarking
- cross-validation
- hyperparameter optimization
- SHAP explainability
- applicability domain analysis

the scaffold validation results strengthen the scientific credibility of the developed QSAR workflow.

Overall, the Hybrid Random Forest model can be considered a robust baseline QSAR model for permeability prediction with demonstrated capability to generalize to novel chemical scaffolds.